In [45]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

In [47]:
class MovieRecommender:
    def __init__(self, movies_df, ratings_df):
        self.movies = movies_df.copy()
        self.ratings = ratings_df.copy()
        self._build_models()

    # =========================================================
    # BUILD MODELS
    # =========================================================
    def _build_models(self):

        # ---------------- CONTENT MODEL ----------------
        self.movies['content'] = (
            self.movies['title'].fillna('') + ' ' +
            self.movies['genres'].fillna('')
        )

        tfidf = TfidfVectorizer(stop_words='english')
        self.tfidf_matrix = tfidf.fit_transform(self.movies['content'])
        self.content_similarity = cosine_similarity(self.tfidf_matrix)

        # ---------------- USER-MOVIE MATRIX ----------------
        self.user_movie_matrix = self.ratings.pivot_table(
            index='userId',
            columns='movieId',
            values='rating'
        ).fillna(0)

        # ---------------- KMEANS CLUSTERING ----------------
        self.kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
        self.user_clusters = self.kmeans.fit_predict(self.user_movie_matrix)

        # map: userId -> cluster
        self.user_cluster_map = pd.Series(
            self.user_clusters,
            index=self.user_movie_matrix.index
        )

        # ---------------- KNN COLLABORATIVE ----------------
        self.knn = NearestNeighbors(metric='cosine', algorithm='brute')
        self.knn.fit(self.user_movie_matrix)

    # =========================================================
    # USER GENRE PROFILE
    # =========================================================
    def _get_user_genre_preferences(self, user_id, top_n=5):

        if user_id not in self.ratings['userId'].values:
            return {}

        user_ratings = self.ratings[self.ratings['userId'] == user_id]

        merged = user_ratings.merge(
            self.movies[['movieId', 'genres']],
            on='movieId'
        )

        genre_ratings = {}

        for _, row in merged.iterrows():
            if pd.notna(row['genres']):
                for g in row['genres'].split('|'):
                    genre_ratings.setdefault(g, []).append(row['rating'])

        genre_avg = {
            g: np.mean(v)
            for g, v in genre_ratings.items()
        }

        return dict(
            sorted(genre_avg.items(), key=lambda x: x[1], reverse=True)[:top_n]
        )

    # =========================================================
    # FALLBACK POPULAR
    # =========================================================
    def _fallback_popular(self, n):

        popular = (
            self.ratings.groupby('movieId')['rating']
            .agg(['mean', 'count'])
        )

        popular['score'] = popular['mean'] * np.log1p(popular['count'])

        top = popular.sort_values('score', ascending=False).head(n)

        result = self.movies[self.movies['movieId'].isin(top.index)][
            ['movieId', 'title', 'genres']
        ].copy()

        result = result.merge(top[['score']], left_on='movieId', right_index=True)

        return result

    # =========================================================
    # CONTENT-BASED
    # =========================================================
    def content_recommend(self, movie_title, n=10):

        if movie_title not in self.movies['title'].values:
            return self._fallback_popular(n)

        idx = self.movies[self.movies['title'] == movie_title].index[0]

        sim_scores = list(enumerate(self.content_similarity[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n*3+1]

        movie_indices = [i[0] for i in sim_scores]
        scores = [i[1] for i in sim_scores]

        result = self.movies.iloc[movie_indices][
            ['movieId', 'title', 'genres']
        ].copy()

        result['score'] = scores

        return result

    # =========================================================
    # COLLABORATIVE (WITH CLUSTERING)
    # =========================================================
    def collaborative_recommend(self, user_id, n=10):

        if user_id not in self.user_movie_matrix.index:
            return self._fallback_popular(n)

        user_vector = self.user_movie_matrix.loc[[user_id]]

        distances, indices = self.knn.kneighbors(
            user_vector,
            n_neighbors=min(6, len(self.user_movie_matrix))
        )

        all_neighbors = self.user_movie_matrix.index[indices.flatten()]

        # ---------------- CLUSTER FILTER ----------------
        user_cluster = self.user_cluster_map.get(user_id, -1)

        cluster_neighbors = [
            u for u in all_neighbors
            if self.user_cluster_map.get(u, -1) == user_cluster
        ]

        if len(cluster_neighbors) < 2:
            cluster_neighbors = all_neighbors

        neighbor_ratings = self.ratings[
            self.ratings['userId'].isin(cluster_neighbors)
        ]

        watched = set(
            self.ratings[self.ratings['userId'] == user_id]['movieId']
        )

        stats = neighbor_ratings.groupby('movieId')['rating'].agg(['mean', 'count'])

        stats['score'] = stats['mean'] * np.log1p(stats['count'])

        stats = stats[~stats.index.isin(watched)]
        stats = stats.sort_values('score', ascending=False).head(n)

        result = self.movies[self.movies['movieId'].isin(stats.index)][
            ['movieId', 'title', 'genres']
        ].copy()

        result = result.merge(stats[['score']], left_on='movieId', right_index=True)

        return result

    # =========================================================
    # HYBRID
    # =========================================================
    def hybrid_recommend(self, user_id, movie_title, n=10):

        content = self.content_recommend(movie_title, n*3)
        collab = self.collaborative_recommend(user_id, n*3)

        if content.empty and collab.empty:
            return self._fallback_popular(n)

        if not content.empty:
            content['score'] = MinMaxScaler().fit_transform(content[['score']])

        if not collab.empty:
            collab['score'] = MinMaxScaler().fit_transform(collab[['score']])

        content['source'] = 'content'
        collab['source'] = 'collaborative'

        hybrid = pd.concat([content, collab], ignore_index=True)

        hybrid = hybrid.groupby(
            ['movieId', 'title', 'genres'],
            as_index=False
        )['score'].sum()

        return hybrid.sort_values('score', ascending=False).head(n)

    # =========================================================
    # EXPLANATION (SAFE)
    # =========================================================
    def explain_recommendation(self, user_id, movie_title, n=5):

        recs = self.hybrid_recommend(user_id, movie_title, n)
        user_genres = self._get_user_genre_preferences(user_id)

        explanations = []
        confidences = []

        for _, movie in recs.iterrows():

            score = movie.get('score', 0.5)

            movie_genres = set(
                movie['genres'].split('|')
            ) if pd.notna(movie['genres']) else set()

            explanation = "Recommended based on hybrid model"
            confidence = float(score)

            common = movie_genres.intersection(user_genres.keys())
            if common:
                explanation = f"You like {', '.join(list(common)[:2])} movies"
                confidence = max(confidence, 0.7)

            explanations.append(explanation)
            confidences.append(confidence)

        recs = recs.copy()
        recs['explanation'] = explanations
        recs['confidence'] = confidences

        return recs.sort_values('confidence', ascending=False)[
            ['title', 'genres', 'explanation', 'confidence']
        ]

    # =========================================================
    # USER PROFILE
    # =========================================================
    def get_user_preferences(self, user_id):

        if user_id not in self.ratings['userId'].values:
            return {'error': 'User not found'}

        user_ratings = self.ratings[self.ratings['userId'] == user_id]

        genre_prefs = self._get_user_genre_preferences(user_id)

        stats = {
            'total_ratings': len(user_ratings),
            'avg_rating': user_ratings['rating'].mean(),
            'rating_std': user_ratings['rating'].std(),
            'favorite_genres': genre_prefs,
            'rating_distribution': user_ratings['rating'].value_counts().to_dict()
        }

        top_movies = user_ratings.nlargest(5, 'rating')

        stats['top_movies'] = top_movies.merge(
            self.movies[['movieId', 'title']],
            on='movieId'
        )[['title', 'rating']].to_dict('records')

        return stats

In [17]:
# Load your data
movies = pd.read_csv('/content/mlfile/movies.csv')
ratings = pd.read_csv('/content/mlfile/ratings.csv')


In [18]:
display(movies.head())

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [19]:
display(ratings.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [49]:
# Initialize recommender
recommender = MovieRecommender(movies, ratings)

# Get content-based recommendations
content_recs = recommender.content_recommend('Toy Story (1995)', n=10)

# Get collaborative recommendations
collab_recs = recommender.collaborative_recommend(user_id=1, n=10)

# Get hybrid recommendations
hybrid_recs = recommender.hybrid_recommend(user_id=1, movie_title='Toy Story (1995)', n=10)

# Get recommendations with explanations
explanations = recommender.explain_recommendation(user_id=1, movie_title='explanations', n=5)

# Get user preferences
preferences = recommender.get_user_preferences(user_id=1)



In [50]:
print("Preferances:")
print(preferences)

Preferances:
{'total_ratings': 232, 'avg_rating': np.float64(4.366379310344827), 'rating_std': 0.8000480467733447, 'favorite_genres': {'Film-Noir': np.float64(5.0), 'Animation': np.float64(4.689655172413793), 'Musical': np.float64(4.681818181818182), 'Children': np.float64(4.5476190476190474), 'Drama': np.float64(4.529411764705882)}, 'rating_distribution': {5.0: 124, 4.0: 76, 3.0: 26, 2.0: 5, 1.0: 1}, 'top_movies': [{'title': 'Seven (a.k.a. Se7en) (1995)', 'rating': 5.0}, {'title': 'Usual Suspects, The (1995)', 'rating': 5.0}, {'title': 'Bottle Rocket (1996)', 'rating': 5.0}, {'title': 'Rob Roy (1995)', 'rating': 5.0}, {'title': 'Canadian Bacon (1995)', 'rating': 5.0}]}


In [51]:
print("Explanation:")
print(explanations)

Explanation:
                               title                          genres  \
5   Shawshank Redemption, The (1994)                     Crime|Drama   
18                     Aliens (1986)  Action|Adventure|Horror|Sci-Fi   
12             Godfather, The (1972)                     Crime|Drama   
8                Blade Runner (1982)          Action|Sci-Fi|Thriller   
6                Forrest Gump (1994)        Comedy|Drama|Romance|War   

                          explanation  confidence  
5               You like Drama movies    1.000000  
18  Recommended based on hybrid model    1.000000  
12              You like Drama movies    0.965381  
8   Recommended based on hybrid model    0.744182  
6               You like Drama movies    0.700000  


In [53]:
print("Content-based Recommendations:")
print(content_recs)

Content-based Recommendations:
      movieId                                              title  \
2355     3114                                 Toy Story 2 (1999)   
7355    78499                                 Toy Story 3 (2010)   
3595     4929                                    Toy, The (1982)   
2539     3400              We're Back! A Dinosaur's Story (1993)   
26         27                                Now and Then (1995)   
4089     5843                                Toy Soldiers (1991)   
1617     2161                      NeverEnding Story, The (1984)   
6194    45074                                   Wild, The (2006)   
1           2                                     Jumanji (1995)   
12         13                                       Balto (1995)   
2227     2961                            Story of Us, The (1999)   
109       126                  NeverEnding Story III, The (1994)   
7039    68954                                          Up (2009)   
3568     4886    

In [54]:
print("collabrative-based Recommendations:")
print(collab_recs)

collabrative-based Recommendations:
      movieId                                              title  \
474       541                                Blade Runner (1982)   
507       589                  Terminator 2: Judgment Day (1991)   
602       750  Dr. Strangelove or: How I Learned to Stop Worr...   
659       858                              Godfather, The (1972)   
706       924                       2001: A Space Odyssey (1968)   
793      1036                                    Die Hard (1988)   
902      1200                                      Aliens (1986)   
922      1221                     Godfather: Part II, The (1974)   
1057     1374             Star Trek II: The Wrath of Khan (1982)   
1211     1610                   Hunt for Red October, The (1990)   

                                genres     score  
474             Action|Sci-Fi|Thriller  8.047190  
507                      Action|Sci-Fi  7.167038  
602                         Comedy|War  6.931472  
659        

In [55]:
print("Hybrid-based Recommendations:")
print(hybrid_recs)

Hybrid-based Recommendations:
    movieId                              title  \
35     1200                      Aliens (1986)   
68     3114                 Toy Story 2 (1999)   
96    78499                 Toy Story 3 (2010)   
22      541                Blade Runner (1982)   
29      858              Godfather, The (1972)   
45     1610   Hunt for Red October, The (1990)   
76     4929                    Toy, The (1982)   
32     1036                    Die Hard (1988)   
24      589  Terminator 2: Judgment Day (1991)   
31      924       2001: A Space Odyssey (1968)   

                                              genres     score  
35                    Action|Adventure|Horror|Sci-Fi  1.000000  
68       Adventure|Animation|Children|Comedy|Fantasy  1.000000  
96  Adventure|Animation|Children|Comedy|Fantasy|IMAX  0.907070  
22                            Action|Sci-Fi|Thriller  0.786649  
29                                       Crime|Drama  0.786649  
45                         Ac

In [42]:
def evaluate_recommender(model, test_ratings, recommender_type='hybrid', k=10):

    precisions = []
    recalls = []
    f1_scores = []
    hit_rates = []
    ndcgs = []

    users = test_ratings['userId'].unique()

    for user_id in users:

        relevant_movies = set(
            test_ratings[
                (test_ratings['userId'] == user_id) &
                (test_ratings['rating'] >= 4)
            ]['movieId']
        )

        if len(relevant_movies) == 0:
            continue

        try:

            if recommender_type == 'content':

                seed_movie_id = list(relevant_movies)[0]

                movie_row = model.movies[
                    model.movies['movieId'] == seed_movie_id
                ]

                if movie_row.empty:
                    continue

                seed_title = movie_row.iloc[0]['title']

                recs = model.content_recommend(
                    seed_title,
                    n=k
                )

            elif recommender_type == 'collaborative':

                recs = model.collaborative_recommend(
                    user_id,
                    n=k
                )

            elif recommender_type == 'hybrid':

                seed_movie_id = list(relevant_movies)[0]

                movie_row = model.movies[
                    model.movies['movieId'] == seed_movie_id
                ]

                if movie_row.empty:
                    continue

                seed_title = movie_row.iloc[0]['title']

                recs = model.hybrid_recommend(
                    user_id,
                    seed_title,
                    n=k
                )

            else:
                continue

        except:
            continue

        recommended = list(recs['movieId'])

        hits = len(
            set(recommended).intersection(relevant_movies)
        )

        precision = hits / k
        recall = hits / len(relevant_movies)

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0
        )

        hit_rate = 1 if hits > 0 else 0

        dcg = sum(
            1 / np.log2(i + 2)
            for i, movie in enumerate(recommended)
            if movie in relevant_movies
        )

        idcg = sum(
            1 / np.log2(i + 2)
            for i in range(min(len(relevant_movies), k))
        )

        ndcg = dcg / idcg if idcg > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        hit_rates.append(hit_rate)
        ndcgs.append(ndcg)

    return {
        'Precision@10': np.mean(precisions),
        'Recall@10': np.mean(recalls),
        'F1@10': np.mean(f1_scores),
        'HitRate': np.mean(hit_rates),
        'NDCG@10': np.mean(ndcgs)
    }

In [43]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

model = MovieRecommender(
    movies_df=movies,
    ratings_df=train_ratings
)
content_results = evaluate_recommender(
    model,
    test_ratings,
    recommender_type='content'
)

collab_results = evaluate_recommender(
    model,
    test_ratings,
    recommender_type='collaborative'
)

hybrid_results = evaluate_recommender(
    model,
    test_ratings,
    recommender_type='hybrid'
)

In [44]:
print("CONTENT-BASED RESULTS")
for metric, value in content_results.items():
    print(f"{metric}: {value:.4f}")

print("\nCOLLABORATIVE RESULTS")
for metric, value in collab_results.items():
    print(f"{metric}: {value:.4f}")

print("\nHYBRID RESULTS")
for metric, value in hybrid_results.items():
    print(f"{metric}: {value:.4f}")

CONTENT-BASED RESULTS
Precision@10: 0.0210
Recall@10: 0.0197
F1@10: 0.0165
HitRate: 0.1653
NDCG@10: 0.0211

COLLABORATIVE RESULTS
Precision@10: 0.1678
Recall@10: 0.1697
F1@10: 0.1329
HitRate: 0.7195
NDCG@10: 0.2043

HYBRID RESULTS
Precision@10: 0.1162
Recall@10: 0.1188
F1@10: 0.0934
HitRate: 0.6194
NDCG@10: 0.1551
